# Smooth back scratcher + massage

Slow, smooth, calligraphy-style moves around your scratch pose: a full session with warm-up, scratching, massage and a calm finish. No recording.

**Every time:** close other arm notebooks → **Kernel → Restart Kernel and Clear Outputs** → run cell 1 and cell 2.

**■ Stop** = it lifts off and goes to sleep. **Arm power plug** = emergency stop.

Do Step 1 and Step 2 with **nobody on the bed**.

## 1. Settings

In [ ]:
import time, math, random, io, contextlib

SIM = False              # True = test on the laptop without moving the arm

# ---- WHERE IT SCRATCHES -------------------------------------------------
# Your pose where the gripper touches your back (from scratch_simple):
SCRATCH_POSE = [180, 31, 149, 135, 95, 150]

# ---- LIFTING OFF THE BACK -----------------------------------------------
# Which servo lifts the gripper AWAY from your back, and which way.
# Run cell "Step 1: check lift" with NOBODY on the bed and watch:
# the gripper must go UP / AWAY from the mattress. If it goes down, flip the sign.
LIFT_SERVO = 2
LIFT_DEG = 15            # try -15 if it goes the wrong way
LIFT_CHECKED = False     # set True after Step 1 looked right. Scratch won't run until then.

# Sleep pose = scratch pose lifted far away. Or put your own numbers, e.g. [90, 120, 30, 30, 90, 150]
SLEEP_LIFT = 45
SLEEP_POSE = None

# ---- FEEL ---------------------------------------------------------------
MINUTES = 10             # then it lifts off and goes to sleep
MAX_SPEED = 25           # degrees per second for any motor. Lower = slower, smoother, quieter (15-40)
STRENGTH = 1.0           # size of all scratch moves. 0.5 = small/gentle, 1.5 = bigger
TOUCH_SOFTEN = 0         # degrees of lift while scratching. 2-4 = lighter touch, 0 = exactly your pose
GRIP_OPEN = 105          # gripper "claw" opening
GRIP_CLOSED = 150

# Spots along the back = how much the base (servo 1) moves from your pose to reach other areas
SPOTS = [0, -6, -12, -6]

# What it does: "mix" = massage + scratching, "massage" = only massage moves, "scratch" = only scratching
MODE = "mix"

# Safety: while scratching, no motor may go further than this from SCRATCH_POSE
MAX_FROM_POSE = 20


## 2. Program + connect (just run it)

In [ ]:
STEP_MS = 40   # one small step every 40 ms -> smooth, calligraphy-like
_LIMITS = {1: 180, 2: 180, 3: 180, 4: 180, 5: 270, 6: 180}

class SimArm:
    def __init__(self):
        self.a = list(SCRATCH_POSE)
        self.log = []
    def Arm_serial_servo_write6_array(self, a, t):
        self.a = list(a); self.log.append(list(a))
    def Arm_serial_servo_read(self, s):
        return self.a[s - 1]

def connect():
    if SIM:
        return SimArm()
    from Arm_Lib import Arm_Device
    a = Arm_Device()
    time.sleep(0.1)
    return a

def lifted(pose, deg):
    p = list(pose)
    p[LIFT_SERVO - 1] += deg
    return p

TOUCH = lifted(SCRATCH_POSE, TOUCH_SOFTEN)
HOVER = lifted(SCRATCH_POSE, LIFT_DEG)
SLEEP = list(SLEEP_POSE) if SLEEP_POSE else lifted(SCRATCH_POSE, SLEEP_LIFT * (1 if LIFT_DEG > 0 else -1))

_cur = None          # where we believe the arm is right now
_guard = False       # True while near your back -> stay inside MAX_FROM_POSE

def _clamp(p):
    out = []
    for i, v in enumerate(p):
        sid = i + 1
        if _guard and sid <= 5:
            c = SCRATCH_POSE[i]
            v = max(c - MAX_FROM_POSE, min(c + MAX_FROM_POSE, v))
        out.append(max(0, min(_LIMITS[sid], v)))
    return out

def _send(p):
    q = [int(round(v)) for v in _clamp(p)]
    arm.Arm_serial_servo_write6_array(q, STEP_MS + 30)
    if not SIM:
        time.sleep(STEP_MS / 1000)

def _quiet_read(sid):
    with contextlib.redirect_stdout(io.StringIO()):
        try:
            v = arm.Arm_serial_servo_read(sid)
        except Exception:
            v = None
    if v is None or not (0 <= v <= _LIMITS[sid]):
        return None
    return v

def where_am_i():
    """Read the real position. If a motor doesn't answer, return None (we never guess)."""
    pose = []
    for sid in range(1, 6):
        v = None
        for _ in range(6):
            v = _quiet_read(sid)
            if v is not None:
                break
            time.sleep(0.02)
        if v is None:
            return None
        pose.append(int(v))
    pose.append(GRIP_CLOSED)
    return pose

def glide(target, at_least_s=0.0):
    """Smooth move: slow start, slow end, never faster than MAX_SPEED."""
    global _cur
    start = list(_cur)
    dist = max(abs(t - s) for t, s in zip(target[:5], start[:5]))
    grip = abs(target[5] - start[5])
    # eased motion peaks at ~1.57x average speed -> include that so the peak stays under MAX_SPEED
    dur = max(at_least_s, 1.6 * dist / MAX_SPEED, 1.6 * grip / 60, 0.2)
    n = max(1, int(dur * 1000 / STEP_MS))
    for i in range(1, n + 1):
        s = (1 - math.cos(math.pi * i / n)) / 2
        _send([a + (b - a) * s for a, b in zip(start, target)])
    _cur = list(target)

def path(fn, seconds, speed=1.0):
    """Follow a smooth pattern fn(t) -> offsets {servo: deg}, t from 0 to 1, starting and ending at 0.
    Duration is stretched automatically if it would be faster than MAX_SPEED."""
    global _cur
    base = list(_cur)
    def pose_at(t):
        p = list(base)
        for sid, off in fn(t).items():
            if sid == "L":
                p[LIFT_SERVO - 1] += _LSIGN * min(6, max(0, off))
            else:
                p[sid - 1] += off * (STRENGTH if sid != 6 else 1)
        return p
    # measure the fastest part of the pattern
    k = 400
    pts = [pose_at(i / k) for i in range(k + 1)]
    worst = max(max(abs(a - b) for a, b in zip(p1[:5], p0[:5])) for p0, p1 in zip(pts, pts[1:]))
    gworst = max(abs(p1[5] - p0[5]) for p0, p1 in zip(pts, pts[1:]))
    need = max(worst * k / (MAX_SPEED * speed), gworst * k / 60)
    seconds = max(seconds, need)
    n = max(1, int(seconds * 1000 / STEP_MS))
    for i in range(1, n + 1):
        _send(pose_at(i / n))
    _cur = pose_at(1.0)

# ---- movement building blocks -----------------------------------------------
# Offsets per servo. Key "L" = lift AWAY from your back (only ever >= 0, so a pattern
# can never press deeper than your scratch pose).
_LSIGN = 1 if LIFT_DEG > 0 else -1

def ramp(t):                     # fade in/out so nothing ever jerks
    return math.sin(math.pi * t) ** 0.5

def wave(t, n):                  # 0 -> 1 -> 0, n times
    return (1 - math.cos(2 * math.pi * n * t)) / 2

def ease(t):
    return (1 - math.cos(math.pi * t)) / 2

# ---- SCRATCH moves ---------------------------------------------------------
def p_strokes(t):        # slow side strokes, like brush strokes
    return {1: -6 * wave(t, 3)}

def p_wrist(t):          # quick little up-down scratches with the wrist
    return {4: 5 * math.sin(2 * math.pi * 6 * t) * ramp(t)}

def p_eight(t):          # figure eight, like handwriting
    a = 2 * math.pi * 2 * t
    return {1: -5 * wave(t, 2), 4: 4 * math.sin(2 * a) * ramp(t)}

def p_claw(t):           # gripper opens/closes like fingernails while sliding
    return {6: (GRIP_OPEN - GRIP_CLOSED) * wave(t, 3), 4: 3 * math.sin(2 * math.pi * 3 * t) * ramp(t)}

def p_zigzag(t):         # fast-ish small zigzags drifting sideways and back
    return {1: -7 * wave(t, 1), 4: 3 * math.sin(2 * math.pi * 10 * t) * ramp(t)}

# ---- MASSAGE moves ---------------------------------------------------------
def p_circles(t):        # slow circles
    a = 2 * math.pi * 3 * t
    return {1: -4 * wave(t, 3), 4: 4 * math.sin(a) * ramp(t)}

def p_spiral(t):         # circles that grow bigger, then smaller again
    r = math.sin(math.pi * t)
    a = 2 * math.pi * 4 * t
    return {1: -6 * r * (1 - math.cos(a)) / 2, 4: 5 * r * math.sin(a)}

def p_knead(t):          # kneading: open + lift a little, come down, squeeze, slow
    n = 3
    ph = (n * t) % 1.0
    grip = (GRIP_OPEN - GRIP_CLOSED) * (math.sin(math.pi * ph) ** 2)
    return {6: grip, "L": 3 * math.sin(math.pi * ph) ** 2 * (1 - ease(ph)), 5: 6 * math.sin(2 * math.pi * n * t) * ramp(t)}

def p_tapping(t):        # gentle tapping: lifts a few degrees and comes back to the touch point
    return {"L": 4 * wave(t, 8), 1: -3 * wave(t, 1)}

def p_roll(t):           # rolling wave: elbow and wrist move against each other
    a = math.sin(2 * math.pi * 3 * t) * ramp(t)
    return {3: 3 * a, 4: -4 * a}

def p_breathe(t):        # slow strokes with soft pressure pulses (lighter / normal)
    return {1: -6 * wave(t, 2), "L": 2.5 * wave(t, 4)}

def p_twist(t):          # rubbing by turning the wrist
    return {5: 12 * math.sin(2 * math.pi * 3 * t) * ramp(t)}

#          name:           (function, seconds, kind)
PATTERNS = {"strokes":      (p_strokes, 7, "scratch"),
            "wrist scratch":(p_wrist,   6, "scratch"),
            "figure eight": (p_eight,   8, "scratch"),
            "claw":         (p_claw,    7, "scratch"),
            "zigzag":       (p_zigzag,  7, "scratch"),
            "circles":      (p_circles, 8, "massage"),
            "spiral":       (p_spiral, 10, "massage"),
            "kneading":     (p_knead,  10, "massage"),
            "tapping":      (p_tapping, 7, "massage"),
            "rolling wave": (p_roll,    8, "massage"),
            "breathing":    (p_breathe, 9, "massage"),
            "twist":        (p_twist,   6, "massage")}

def pick(n):
    names = [k for k, v in PATTERNS.items() if MODE == "mix" or v[2] == MODE]
    return random.sample(names, min(n, len(names)))

# ---- start / stop ----------------------------------------------------------
def start_safely():
    """Find out where the arm really is, then glide slowly to HOVER (above the back)."""
    global _cur, _guard
    _guard = False
    pose = where_am_i()
    if pose is None:
        raise SystemExit("Can't read the arm position. Restart the kernel, check the USB "
                         "connection in VMware, and try again. (The arm did not move.)")
    _cur = pose
    print("Arm is at", pose[:5], "- gliding slowly to the start...")
    glide(HOVER, at_least_s=3)

def go_to_sleep():
    global _guard
    _guard = False
    if _cur is None:
        start_safely()
    print("Lifting off and going to sleep...")
    glide(HOVER)
    glide(SLEEP, at_least_s=3)
    print("Sleeping.")

def touch_down(spot):
    global _guard
    above = list(HOVER); above[0] += spot
    down = list(TOUCH);  down[0] += spot
    glide(above)
    _guard = True
    glide(down, at_least_s=1.5)

def travel(from_spot, to_spot):
    """Glide along the back to the next spot with small zigzags (no lifting)."""
    d = to_spot - from_spot
    path(lambda t: {1: d * ease(t), 4: 3 * math.sin(2 * math.pi * 4 * t) * ramp(t),
                    "L": 1.5 * wave(t, 4)}, 5)

def lift_off():
    global _guard
    up = list(_cur); up[LIFT_SERVO - 1] = HOVER[LIFT_SERVO - 1]
    glide(up, at_least_s=1.0)
    _guard = False

# ---- the cells call these --------------------------------------------------
def check_lift():
    """NOBODY on the bed. Goes to your scratch pose, then lifts. Watch the direction."""
    global _cur
    pose = where_am_i()
    if pose is None:
        raise SystemExit("Can't read the arm position - check USB and restart the kernel.")
    _cur = pose
    print("Gliding to your SCRATCH pose (where your back would be)...")
    glide(SCRATCH_POSE, at_least_s=3)
    time.sleep(0 if SIM else 2)
    print(f"Now LIFTING (servo {LIFT_SERVO}, {LIFT_DEG:+} deg). It must move AWAY from the mattress!")
    glide(HOVER, at_least_s=2)
    time.sleep(0 if SIM else 2)
    print("If it went UP/AWAY: set LIFT_CHECKED = True in cell 1.")
    print("If it went DOWN/INTO the bed: change LIFT_DEG to the opposite sign, run cells 1+2, and check again.")

def demo_in_air():
    """Shows every pattern in the AIR (at hover height), nobody on the bed."""
    try:
        start_safely()
        for name, (fn, sec, kind) in PATTERNS.items():
            print("  pattern:", name)
            path(fn, sec)
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        go_to_sleep()

def scratch():
    if not LIFT_CHECKED:
        raise SystemExit("First run 'Step 1: check lift' and set LIFT_CHECKED = True in cell 1.")
    end = time.time() + MINUTES * 60
    left = lambda: end - time.time()
    try:
        start_safely()
        print(f"Session: {MINUTES} min, mode '{MODE}'. Press Stop once to end.")
        # 1) warm-up: slow long strokes across the whole back
        touch_down(SPOTS[0])
        print("  warm-up: long slow strokes")
        span = min(SPOTS) - SPOTS[0]
        path(lambda t: {1: span * wave(t, 2), 4: 2 * math.sin(2 * math.pi * 6 * t) * ramp(t)}, 16, speed=0.6)
        here = SPOTS[0]
        rounds = 0
        # 2) main part: every spot gets 2-3 different moves, then travel to the next
        while left() > 25:
            for spot in SPOTS:
                if left() <= 25:
                    break
                if spot != here:
                    print(f"  -> travelling to spot {spot}")
                    travel(here, spot)
                    here = spot
                for name in pick(random.choice([2, 3])):
                    if left() <= 25:
                        break
                    fn, sec, kind = PATTERNS[name]
                    print(f"  spot {spot}: {name}")
                    path(fn, sec)
            rounds += 1
            if SIM and rounds >= 1:
                break
        # 3) finish: gentle tapping, then one calm stroke
        print("  finish: tapping + calm stroke")
        path(p_tapping, 6)
        path(lambda t: {1: -6 * wave(t, 1)}, 8, speed=0.5)
        print("Done.")
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        try:
            if _guard:
                lift_off()
        finally:
            go_to_sleep()

arm = connect()
print("Connected" if not SIM else "Simulation mode")
print("Scratch pose:", SCRATCH_POSE, " hover:", HOVER, " sleep:", SLEEP)


## Step 1: check lift direction (nobody on the bed)
It glides to your scratch pose, then lifts. The gripper must move **away** from the mattress. Then set `LIFT_CHECKED = True` in cell 1 and run cells 1 and 2 again.

In [ ]:
check_lift()

## Step 2: demo in the air (nobody on the bed)
Shows all 12 moves above the bed so you can see and hear them.

In [ ]:
demo_in_air()

## Step 3: Scratch!
Test with your **hand** on the bed first, then your back.

In [ ]:
scratch()

## Sleep now

In [ ]:
go_to_sleep()

## Voice control (Google Home)
Run this cell once (after cells 1 and 2), then run **Listen**. Say *"Hey Google, activate massage"* etc. **■ Stop** ends listening and parks the arm.

Test without the speaker, in a VM terminal: `curl -d massage ntfy.sh/YOUR_TOPIC`

In [ ]:
import threading, urllib.request, json as _json

# Your private "mailbox" name on ntfy.sh. Must be EXACTLY the same in the IFTTT web request URL.
# Keep it secret - anyone who knows it could start the arm.
NTFY_TOPIC = "backarm-4d4e72eb9e"

_abort = False
_worker = None
_orig_send = _send

def _send(p):                       # every small step checks for a "stop"
    global _abort
    if _abort:
        _abort = False
        raise KeyboardInterrupt
    _orig_send(p)

def _run(mode):
    global MODE
    MODE = mode
    try:
        scratch()
    except SystemExit as e:
        print("Not started:", e)

def busy():
    return _worker is not None and _worker.is_alive()

def start(mode):
    global _worker
    if busy():
        print("Already running - say 'stop' first.")
        return
    _worker = threading.Thread(target=_run, args=(mode,), daemon=True)
    _worker.start()

def stop():
    global _abort
    if busy():
        _abort = True

def handle(word):
    global MAX_SPEED
    print(f"[voice] '{word}'")
    if "stop" in word or "sleep" in word:
        stop()
    elif "massage" in word or "massege" in word:
        start("massage")
    elif "scratch" in word:
        start("scratch")
    elif "mix" in word or "back" in word:
        start("mix")
    elif "slow" in word:
        MAX_SPEED = max(10, MAX_SPEED * 0.75); print("  speed now", round(MAX_SPEED))
    elif "fast" in word:
        MAX_SPEED = min(40, MAX_SPEED * 1.25); print("  speed now", round(MAX_SPEED))
    else:
        print("  (unknown command)")

def listen():
    url = f"https://ntfy.sh/{NTFY_TOPIC}/json"
    print("Listening for voice commands... (press Stop button to quit)")
    try:
        while True:
            try:
                with urllib.request.urlopen(url, timeout=90) as r:
                    for line in r:
                        msg = _json.loads(line)
                        if msg.get("event") == "message":
                            handle(msg.get("message", "").strip().lower())
            except KeyboardInterrupt:
                raise
            except Exception as e:
                print("  connection problem, retrying in 5 s:", e)
                time.sleep(5)
    except KeyboardInterrupt:
        print("Quitting listener - stopping the arm.")
        stop()
        if _worker:
            _worker.join(timeout=60)

print("Voice ready. Topic:", NTFY_TOPIC)


## Listen

In [ ]:
listen()